# 02 · Severity from PlantSeg masks
### *From Diagnosis to Decision* — ICA 2026

The **severity** component of the paper — the bridge from *diagnosis* to
*intervention tier* — needs pixel-level ground truth. PlantSeg provides it:
in-the-wild images with per-lesion segmentation masks. This notebook:

1. loads the PlantSeg images, masks and `Metadata.csv`;
2. sanity-checks the masks;
3. derives per-image **severity = lesion area ÷ leaf area** (leaf area via a
   zero-cost Otsu leaf/background segmentation);
4. bins severity into intervention tiers and reports the distribution.

**Which Zenodo record.** Use **v7 — record `17719108`, DOI
`10.5281/zenodo.17719108`** (the version the peer-reviewed *Scientific Data*
paper cites; `~1.1 GB`, **CC BY-NC 4.0**). Records `13762907` (v2) and
`13958858` (v4) are superseded; `13293891` (v1) is access-restricted. The
earlier versions were **CC BY-NC-ND 4.0** — cite the license of the exact
version you download.

## 1 · Setup & locate PlantSeg

In [ ]:
# --- Environment config: works on Google Colab AND locally --------------------
import os, sys, pathlib

def in_colab():
    return "google.colab" in sys.modules or os.path.exists("/content")

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/diagnosis-to-decision")
else:
    # local fallback: repo root (edit if you cloned elsewhere)
    PROJECT_ROOT = pathlib.Path(
        os.environ.get("ICA_PROJECT_ROOT", pathlib.Path.cwd().parents[0])
    )

DATA_RAW     = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_MAPPING = PROJECT_ROOT / "data" / "mapping"
FIGDIR       = PROJECT_ROOT / "reports" / "figures"
for p in (DATA_RAW, DATA_INTERIM, DATA_MAPPING, FIGDIR):
    p.mkdir(parents=True, exist_ok=True)

print("Colab:", in_colab())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW exists:", DATA_RAW.exists())

In [ ]:
import numpy as np, pandas as pd, pathlib
import matplotlib.pyplot as plt
from PIL import Image
from skimage.filters import threshold_otsu
from skimage.color import rgb2hsv

PS = DATA_RAW / "plantseg"
have = PS.exists() and any(PS.rglob("*"))
print("PlantSeg present:", have, "-", PS)
if not have:
    print("Download first (nb 00). Kaggle:  kaggle datasets download -d weitianqi/plantseg")
    print("or Zenodo v7:  https://zenodo.org/records/17719108")

In [ ]:
# Locate images, masks/annotations and metadata robustly (layout varies by version)
def find_one(patterns):
    for pat in patterns:
        hits = list(PS.rglob(pat))
        if hits: return hits
    return []

meta_files = find_one(["Metadata.csv","metadata.csv","*metadata*.csv"])
img_dirs   = sorted({p.parent for p in PS.rglob("*.jpg")})
ann_dirs   = sorted({p.parent for p in PS.rglob("*.png")})
print("metadata:", meta_files[:1])
print("candidate image dirs:", [str(d.relative_to(PS)) for d in img_dirs][:6])
print("candidate mask  dirs:", [str(d.relative_to(PS)) for d in ann_dirs][:6])
meta = pd.read_csv(meta_files[0]) if meta_files else None
if meta is not None:
    print("\nMetadata columns:", list(meta.columns)); display(meta.head())

## 2 · Mask sanity checks

Confirm masks align with images and are non-trivial (a mask that is all-zero or
all-one is unusable). We pair each image with its mask by stem and report
coverage statistics.

In [ ]:
def load_pairs(limit=None):
    """Pair image <-> mask by filename stem."""
    imgs = {p.stem: p for p in PS.rglob("*.jpg")}
    masks = {p.stem: p for p in PS.rglob("*.png")}
    stems = sorted(set(imgs) & set(masks))
    if limit: stems = stems[:limit]
    return [(imgs[s], masks[s]) for s in stems]

pairs = load_pairs()
print(f"paired image/mask files: {len(pairs)}")
def mask_fg_fraction(mask_path):
    m = np.asarray(Image.open(mask_path))
    if m.ndim == 3: m = m[...,0]
    return float((m > 0).mean())
if pairs:
    fracs = np.array([mask_fg_fraction(mp) for _,mp in pairs[:500]])
    print(f"lesion-mask foreground fraction (first 500): "
          f"median {np.median(fracs):.3f}, "
          f"all-zero {np.mean(fracs==0):.1%}, all-one {np.mean(fracs>=0.99):.1%}")

## 3 · Derive severity = lesion ÷ leaf

The mask gives **lesion** pixels. For **leaf** pixels we separate leaf from
background with Otsu on the HSV saturation/green response (the zero-cost
baseline from the spec). Severity = lesion_px / leaf_px, clipped to [0, 1].
Where PlantSeg already provides a leaf mask, prefer it and skip the estimate.

In [ ]:
def leaf_mask_otsu(img):
    """Crude leaf/background split: greenish, saturated pixels = leaf."""
    a = np.asarray(img.convert("RGB"), dtype=np.float32)/255.0
    hsv = rgb2hsv(a)
    green = (a[...,1] - 0.5*(a[...,0]+a[...,2]))   # excess green
    sat = hsv[...,1]
    score = 0.5*(green - green.min())/(np.ptp(green)+1e-6) + 0.5*sat
    try:
        thr = threshold_otsu(score)
    except Exception:
        thr = score.mean()
    return score > thr

def severity_of(img_path, mask_path):
    img = Image.open(img_path).convert("RGB")
    m = np.asarray(Image.open(mask_path))
    lesion = (m[...,0] if m.ndim==3 else m) > 0
    leaf = leaf_mask_otsu(img) | lesion         # lesions are part of the leaf
    leaf_px = int(leaf.sum())
    if leaf_px == 0: return np.nan
    return float(lesion.sum()) / leaf_px

if pairs:
    sub = pairs[:800]                            # sample for speed; raise on Colab
    sev = pd.DataFrame({
        "stem":[ip.stem for ip,_ in sub],
        "severity":[severity_of(ip,mp) for ip,mp in sub],
    }).dropna()
    sev.to_csv(DATA_INTERIM/"plantseg_severity.csv", index=False)
    print("severity computed for", len(sev), "images")
    display(sev["severity"].describe().to_frame().T)

## 4 · Intervention tiers

Map continuous severity to the ordinal tiers the action model consumes. These
cut-points are a **starting proposal** — in the paper, calibrate them against the
human-labelled severity sets (notebook `03`) and against agronomic guidance,
and justify the boundaries.

In [ ]:
BINS   = [0, 0.05, 0.15, 0.35, 1.01]
TIERS  = ["trace (monitor)", "mild (spot-treat)",
          "moderate (treat)", "severe (treat + isolate)"]
if pairs and len(sev):
    sev["tier"] = pd.cut(sev["severity"], bins=BINS, labels=TIERS, include_lowest=True)
    dist = sev["tier"].value_counts().reindex(TIERS)
    fig,ax = plt.subplots(1,2, figsize=(13,4))
    ax[0].hist(sev["severity"], bins=30, color="#54A24B"); ax[0].set_title("severity (lesion/leaf)")
    ax[0].set_xlabel("severity"); ax[0].set_ylabel("images")
    for b in BINS[1:-1]: ax[0].axvline(b, ls="--", c="grey", lw=1)
    ax[1].bar(range(len(dist)), dist.values, color="#4C78A8")
    ax[1].set_xticks(range(len(dist))); ax[1].set_xticklabels(TIERS, rotation=25, ha="right")
    ax[1].set_title("intervention-tier distribution")
    plt.tight_layout(); fig.savefig(FIGDIR/"plantseg_severity.png", bbox_inches="tight"); plt.show()
    display(dist.to_frame("images"))

## 5 · Qualitative check
Overlay lesion masks on a few images to confirm the severity numbers are sane.

In [ ]:
if pairs:
    show = pairs[:6]
    fig,axes = plt.subplots(2,3, figsize=(13,8))
    for ax,(ip,mp) in zip(axes.ravel(), show):
        img = np.asarray(Image.open(ip).convert("RGB"))
        m = np.asarray(Image.open(mp)); m = (m[...,0] if m.ndim==3 else m)>0
        ov = img.copy(); ov[m] = [255,0,0]
        blend = (0.6*img + 0.4*ov).astype("uint8")
        ax.imshow(blend); ax.set_title(f"sev={severity_of(ip,mp):.2f}", fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()

---
### For the paper
- Severity is **derived**, not human-labelled — validate it (nb `03`) and report
  the agreement (Spearman ρ) before using it as ground truth.
- The Otsu leaf estimate is a **baseline**; a learned leaf-segmentation model
  would tighten it. State the method and its error mode (thin/occluded leaves).
- Cite **PlantSeg v7 (Zenodo 17719108), CC BY-NC 4.0** — the exact version used.

**Next:** `03_severity_validation_coffee.ipynb`.